In [2]:

# Developer: Dan Stenzel
# Date: October 2026
# Description: Interactive Grazioso Salvare rescue-animal dashboard.

from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
import os

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

# -----------------------------
# Database connection
# -----------------------------
username = "aacuser"
password = "AACuser2026!"

db = AnimalShelter(username, password)

# Retrieve all animal records
df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=["_id"], errors="ignore", inplace=True)

# -----------------------------
# Dashboard branding
# -----------------------------
# Change this path if your logo is in another directory.
image_filename = "Grazioso Salvare Logo.png"

if not os.path.exists(image_filename):
    raise FileNotFoundError(
        f"Logo not found: {image_filename}. "
        "Update image_filename to the correct logo path."
    )

with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

# -----------------------------
# Rescue filter queries
# -----------------------------
def get_rescue_query(filter_type):
    """Return the MongoDB query for the selected rescue category."""

    if filter_type == "water":
        return {
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26, "$lte": 156
            }
        }

    elif filter_type == "mountain":
        return {
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26, "$lte": 156
            }
        }

    elif filter_type == "disaster":
        return {
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20, "$lte": 300
            }
        }

    # Reset returns every animal without filtering.
    return {}


# -----------------------------
# Dashboard layout
# -----------------------------
app = JupyterDash(__name__)

app.layout = html.Div(
    style={"padding": "15px"},
    children=[
        html.Div(
            style={
                "display": "flex",
                "alignItems": "center",
                "gap": "25px"
            },
            children=[
                html.A(
                    html.Img(
                        src="data:image/png;base64," + encoded_image,
                        style={"width": "200px", "height": "auto"}
                    ),
                    href="https://www.snhu.edu",
                    target="_blank"
                ),
                html.Div([
                    html.H1("Grazioso Salvare Rescue Dashboard"),
                    html.H4("Developer: Dan Stenzel")
                ])
            ]
        ),

        html.Hr(),

        html.H3("Select a Rescue Category"),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset / All Animals", "value": "reset"},
                {"label": "Water Rescue", "value": "water"},
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "mountain"
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "disaster"
                }
            ],
            value="reset",
            labelStyle={
                "display": "inline-block",
                "marginRight": "20px",
                "marginBottom": "10px"
            }
        ),

        html.Hr(),

        html.H3("Animal Records"),

        dash_table.DataTable(
            id="datatable-id",
            columns=[
                {"name": column, "id": column}
                for column in df.columns
            ],
            data=df.to_dict("records"),
            page_size=10,
            sort_action="native",
            filter_action="native",
            row_selectable="single",
            selected_rows=[],
            style_table={"overflowX": "auto"},
            style_cell={
                "textAlign": "left",
                "padding": "8px",
                "minWidth": "100px",
                "maxWidth": "250px",
                "whiteSpace": "normal"
            },
            style_header={
                "fontWeight": "bold",
                "backgroundColor": "#D2F3FF"
            },
            style_data_conditional=[]
        ),

        html.Br(),
        html.Hr(),

        html.Div(
            style={
                "display": "flex",
                "flexWrap": "wrap",
                "gap": "20px"
            },
            children=[
                html.Div(
                    id="graph-id",
                    style={"flex": "1", "minWidth": "350px"}
                ),
                html.Div(
                    id="map-id",
                    style={"flex": "1", "minWidth": "350px"}
                )
            ]
        )
    ]
)


# -----------------------------
# Controller: filter the table
# -----------------------------
@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):
    query = get_rescue_query(filter_type)

    records = db.read(query)
    filtered_df = pd.DataFrame.from_records(records)

    filtered_df.drop(
        columns=["_id"],
        errors="ignore",
        inplace=True
    )

    # Keep the original table columns, even if no records match.
    return filtered_df.reindex(columns=df.columns).to_dict("records")


# -----------------------------
# Controller: update breed chart
# -----------------------------
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(view_data):
    if not view_data:
        return html.P("No animals match this selection.")

    chart_df = pd.DataFrame.from_records(view_data)

    if "breed" not in chart_df.columns:
        return html.P("Breed information is unavailable.")

    breed_counts = (
        chart_df["breed"]
        .fillna("Unknown")
        .value_counts()
        .rename_axis("Breed")
        .reset_index(name="Count")
    )

    figure = px.pie(
        breed_counts,
        names="Breed",
        values="Count",
        title="Animal Breeds in Current Results"
    )

    return dcc.Graph(figure=figure)


# -----------------------------
# Controller: update geolocation map
# -----------------------------
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_map(view_data):
    if not view_data:
        return html.P("No animal locations to display.")

    map_df = pd.DataFrame.from_records(view_data)

    required_columns = [
        "location_lat",
        "location_long"
    ]

    if not all(
        column in map_df.columns
        for column in required_columns
    ):
        return html.P(
            "The dataset is missing latitude or longitude columns."
        )

    map_df["location_lat"] = pd.to_numeric(
        map_df["location_lat"], errors="coerce"
    )
    map_df["location_long"] = pd.to_numeric(
        map_df["location_long"], errors="coerce"
    )

    map_df = map_df.dropna(
        subset=["location_lat", "location_long"]
    )

    if map_df.empty:
        return html.P("No valid animal coordinates are available.")

    # Limit the number of markers to keep the map responsive.
    map_df = map_df.head(500)

    markers = []

    for _, animal in map_df.iterrows():
        breed = str(animal.get("breed", "Unknown breed"))
        name = str(animal.get("name", "Unknown name"))

        markers.append(
            dl.Marker(
                position=[
                    float(animal["location_lat"]),
                    float(animal["location_long"])
                ],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup([
                        html.B(name),
                        html.P(breed)
                    ])
                ]
            )
        )

    return dl.Map(
        style={"width": "100%", "height": "500px"},
        center=[30.2672, -97.7431],
        zoom=9,
        children=[
            dl.TileLayer(),
            *markers
        ]
    )


# -----------------------------
# Controller: highlight selected columns
# -----------------------------
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):
    selected_columns = selected_columns or []

    return [
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


# -----------------------------
# Run the dashboard
# -----------------------------
app.run_server(mode="inline", debug=False)


 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [16/Oct/2026 17:46:31] "GET /_alive_38791604-cec9-40b9-9bfe-c1d11947d013 HTTP/1.1" 200 -


127.0.0.1 - - [16/Oct/2026 17:46:31] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:31] "GET /_dash-component-suites/dash/deps/polyfill@7.v2_8_1m1752168217.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:31] "GET /_dash-component-suites/dash/deps/react-dom@16.v2_8_1m1752168217.14.0.min.js HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:31] "GET /_dash-component-suites/dash/html/dash_html_components.v2_0_8m1752168217.min.js HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:31] "GET /_dash-component-suites/dash/deps/prop-types@15.v2_8_1m1752168217.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:31] "GET /_dash-component-suites/dash_leaflet/dash_leaflet.v0_1_23m1752168237.min.js HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:32] "GET /_dash-component-suites/dash/deps/react@16.v2_8_1m1752168217.14.0.min.js HTTP/1.1" 200 -
127.0.0.1 - - [16/Oct/2026 17:46:32] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v2_8_0m1752168217.js HTTP/1.1" 